# Week 5 — Baseline (MNIST Conv-VAE + Digit Classifier + Thickness Probe)
Reproducible runs: fixed seeds, config snapshot, run_id, checkpoints, plots, results.csv, 1-page memo.

## 0. Necessary imports 

In [1]:
# If needed (Colab/Jupyter). Comment out if you already have deps.
!pip install -q torch torchvision pyyaml tqdm matplotlib pandas scikit-learn

In [2]:
import os, json, time, random
from dataclasses import dataclass
from typing import Optional, Dict, Any, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score

## 1. Config params

In [3]:
cfg: Dict[str, Any] = {
    "seed": 42, # to reproduce the results
    "device": "cuda",          # auto-fallback to cpu if no cuda
    "run": {"root": "runs", 
            "tag": "week5"}, # working directory
    "data": {"batch_size": 128, 
             "num_workers": 0, 
             "subset": None},  # subset: e.g. 20000
    "vae":  {"latent_dim": 16, 
             "lr": 1e-3, 
             "epochs": 20, 
             "beta": 1.0, 
             "recon_loss": "bce", 
             "ckpt_every": 5}, # vae params
    "clf":  {"lr": 1e-3, 
             "epochs": 5}, # clf params
    "probe":{"lr": 1e-3, 
             "epochs": 5, 
             "aug": {"thin_k": 1, "thick_k": 1, "blend": 0.2} # for image augmentation
            }, # probe params
}

def get_device(name: str) -> torch.device:
    if name == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_run_dir(root: str, tag: str) -> str:
    ts = time.strftime("%Y%m%d_%H%M%S")
    run_id = f"{ts}_{tag}"
    run_dir = os.path.join(root, run_id)
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=False)
    os.makedirs(os.path.join(run_dir, "plots"), exist_ok=True)
    return run_dir

def save_json(path: str, obj: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

set_seed(int(cfg["seed"]))
device = get_device(cfg["device"])
run_dir = make_run_dir(cfg["run"]["root"], cfg["run"]["tag"])
save_json(os.path.join(run_dir, "config.json"), cfg)

print("device:", device)
print("run_dir:", run_dir)

device: cuda
run_dir: runs/20260224_074746_week5


## 2. Data loading

In [4]:
def seed_worker(worker_id: int):
    # Make DataLoader workers deterministic
    worker_seed = (torch.initial_seed() + worker_id) % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def get_mnist_loaders(batch_size: int, num_workers: int, subset: Optional[int] = None):
    tfm = transforms.ToTensor()
    train = datasets.MNIST(root="data", train=True, download=True, transform=tfm)
    test  = datasets.MNIST(root="data", train=False, download=True, transform=tfm)
    if subset is not None:
        train = Subset(train, list(range(int(subset))))

    g = torch.Generator()
    g.manual_seed(int(cfg["seed"]))
    train_loader = DataLoader(
        train, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True,
        worker_init_fn=seed_worker, generator=g
    )
    test_loader = DataLoader(
        test, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        worker_init_fn=seed_worker, generator=g
    )
    return train_loader, test_loader

train_loader, test_loader = get_mnist_loaders(cfg["data"]["batch_size"], cfg["data"]["num_workers"], cfg["data"]["subset"])

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 518kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.69MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.73MB/s]


In [5]:
def binarize(x: torch.Tensor, thr: float = 0.5) -> torch.Tensor:
    # Turn grayscale image into a binary mask: 1 where pixel > thr, else 0
    return (x > thr).float()

def dilate(x: torch.Tensor, iters: int = 1, k: int = 3) -> torch.Tensor:
    # Morphological dilation for binary images using max-pooling:
    # expands white (1) regions by taking local maximum in kxk window.
    y = x
    pad = k // 2
    for _ in range(iters):
        y = F.max_pool2d(y, kernel_size=k, stride=1, padding=pad)
    return y

def erode(x: torch.Tensor, iters: int = 1, k: int = 3) -> torch.Tensor:
    # Morphological erosion for binary images:
    # shrink white (1) regions. Implemented via dilation of the inverted image:
    # erode(x) = 1 - dilate(1 - x)
    y = x
    for _ in range(iters):
        y = 1.0 - dilate(1.0 - y, iters=1, k=k)
    return y

def apply_thickness_aug(x: torch.Tensor, thin_k: int, thick_k: int, blend: float = 0.3):
    """
    Grayscale thickness augmentation (no binarize).
    blend: доля оригинала, чтобы цифры не "умирали" по яркости.
    Returns:
      out: [B,1,28,28] in [0,1]
      y:   [B], 0=thin, 1=thick
    """
    B = x.size(0)
    y = torch.zeros(B, device=x.device, dtype=torch.long)

    perm = torch.randperm(B, device=x.device)
    half = B // 2
    thick_idx = perm[:half]
    thin_idx  = perm[half:]

    def gray_dilate(img, iters=1, k=3):
        out = img
        for _ in range(iters):
            out = F.max_pool2d(out, kernel_size=k, stride=1, padding=k//2)
        return out

    def gray_erode(img, iters=1, k=3):
        out = img
        for _ in range(iters):
            out = -F.max_pool2d(-out, kernel_size=k, stride=1, padding=k//2)
        return out

    out = x.clone()

    thick = gray_dilate(x[thick_idx], iters=thick_k, k=3)
    thin  = gray_erode(x[thin_idx],  iters=thin_k,  k=3)

    # --- ключевой фикс: нормализация яркости по каждому изображению ---
    eps = 1e-6
    thick = thick / (thick.amax(dim=(1,2,3), keepdim=True) + eps)
    thin  = thin  / (thin.amax(dim=(1,2,3), keepdim=True)  + eps)

    # --- чтобы цифры оставались читаемыми: небольшой blend с оригиналом ---
    thick = (1 - blend) * thick + blend * x[thick_idx]
    thin  = (1 - blend) * thin  + blend * x[thin_idx]

    out[thick_idx] = thick.clamp(0, 1)
    out[thin_idx]  = thin.clamp(0, 1)

    y[thick_idx] = 1
    y[thin_idx] = 0
    return out, y
    

In [6]:
@torch.no_grad()
def save_aug_preview(loader, out_png: str, n: int = 16):
    # Take one batch from the loader
    x, _ = next(iter(loader))
    x = x[:n].to(device)
    # Apply thickness augmentation
    x_aug, y = apply_thickness_aug(
        x, thin_k=cfg["probe"]["aug"]["thin_k"], thick_k=cfg["probe"]["aug"]["thick_k"], 
        blend = cfg["probe"]["aug"]["blend"]
    )
    # Move back to CPU for plotting and remove channel dimension [1]
    x = x.cpu().squeeze(1)
    x_aug = x_aug.cpu().squeeze(1)

    # Plot grid
    cols = n
    rows = 2
    plt.figure(figsize=(cols, rows))
    for i in range(n):
        plt.subplot(rows, cols, i+1); plt.imshow(x[i], cmap="gray"); plt.axis("off")
        plt.subplot(rows, cols, n+i+1); plt.imshow(x_aug[i], cmap="gray"); plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

save_aug_preview(test_loader, os.path.join(run_dir, "plots", "thickness_aug_preview.png"))
print(f'thickness_aug_preview.png saved in {run_dir}/plots')

thickness_aug_preview.png saved in runs/20260224_074746_week5/plots


In [7]:
class ConvEncoder(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()
        # Encoder: map image x [B,1,28,28] -> feature map [B,64,7,7]
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1),  # 28->14
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1), # 14->7
            nn.ReLU(inplace=True),
        )
        # Two heads: mean and log-variance of q(z|x)
        self.fc_mu = nn.Linear(64*7*7, latent_dim)
        self.fc_lv = nn.Linear(64*7*7, latent_dim)

    def forward(self, x):
        # Encode image into a vector, then predict (mu, logvar)
        h = self.net(x).flatten(1)
        return self.fc_mu(h), self.fc_lv(h)

class ConvDecoder(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()
        # expand latent vector z [B,latent_dim] -> [B,64,7,7]
        self.fc = nn.Linear(latent_dim, 64*7*7)
         # Decoder: upsample back to image size 28x28
        self.net = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1), # 7->14
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 1, 4, 2, 1),  # 14->28
        )

    def forward(self, z):
        # Decode latent z into image logits
        h = self.fc(z).view(z.size(0), 64, 7, 7)
        return self.net(h)  # logits

class ConvVAE(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()
        self.enc = ConvEncoder(latent_dim)
        self.dec = ConvDecoder(latent_dim)

    def reparam(self, mu, logvar):
         # Reparameterization trick:
        # z = mu + sigma * eps, where eps ~ N(0, I), sigma = exp(0.5*logvar)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
         # Encode -> sample z -> decode
        mu, lv = self.enc(x)
        z = self.reparam(mu, lv)
        x_logits = self.dec(z)
        return x_logits, mu, lv

def vae_loss(x, x_logits, mu, logvar, beta: float = 1.0, recon: str = "bce"):
    # Reconstruction term: how well we can rebuild x from z
    if recon == "bce":
        recon_loss = F.binary_cross_entropy_with_logits(x_logits, x, reduction="sum") / x.size(0)
    else:
        recon_loss = F.mse_loss(torch.sigmoid(x_logits), x, reduction="sum") / x.size(0)

     # KL term for diagonal Gaussian q(z|x) vs standard normal p(z)=N(0,I)
    kl = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)).mean()
    total = recon_loss + beta * kl
    return total, recon_loss.detach(), kl.detach()

In [8]:
@torch.no_grad()
def save_recon_grid(model, loader, out_png: str, n: int = 16):
    # Saves a grid image: top row = original inputs, bottom row = VAE reconstructions.
    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(device)
    x_logits, _, _ = model(x)
    x_hat = torch.sigmoid(x_logits)

    grid = torch.cat([x.cpu(), x_hat.cpu()], dim=0).squeeze(1)  # [2n,28,28]
    cols, rows = n, 2
    plt.figure(figsize=(cols, rows))
    for i in range(2*n):
        plt.subplot(rows, cols, i+1)
        plt.imshow(grid[i], cmap="gray")
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


def train_vae():
    # Trains a Conv-VAE on MNIST, logs losses, saves checkpoints and reconstruction previews.
    # Create model and optimizer
    model = ConvVAE(latent_dim=int(cfg["vae"]["latent_dim"])).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=float(cfg["vae"]["lr"]))

    beta = float(cfg["vae"]["beta"])
    recon = cfg["vae"]["recon_loss"]
    epochs = int(cfg["vae"]["epochs"])
    ckpt_every = int(cfg["vae"]["ckpt_every"])

    hist = {"loss": [], "recon": [], "kl": [], "val_loss": []}

    for ep in range(1, epochs + 1):
        model.train()
        tot, tot_r, tot_kl = 0.0, 0.0, 0.0
        for x, _ in tqdm(train_loader, desc=f"VAE ep{ep}", leave=False):
            x = x.to(device)
            x_logits, mu, lv = model(x)
            loss, r, kl = vae_loss(x, x_logits, mu, lv, beta=beta, recon=recon)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            tot += loss.item()
            tot_r += r.item()
            tot_kl += kl.item()

        hist["loss"].append(tot / len(train_loader))
        hist["recon"].append(tot_r / len(train_loader))
        hist["kl"].append(tot_kl / len(train_loader))

        model.eval()
        vtot = 0.0
        with torch.no_grad():
            for x, _ in test_loader:
                x = x.to(device)
                x_logits, mu, lv = model(x)
                loss, _, _ = vae_loss(x, x_logits, mu, lv, beta=beta, recon=recon)
                vtot += loss.item()
        hist["val_loss"].append(vtot / len(test_loader))

        # save curves (overwrite = clean)
        plt.figure()
        plt.plot(hist["loss"], label="train")
        plt.plot(hist["val_loss"], label="val")
        plt.legend()
        plt.xlabel("epoch"); plt.ylabel("loss")
        plt.tight_layout()
        plt.savefig(os.path.join(run_dir, "plots", "vae_loss_curve.png"), dpi=150)
        plt.close()

        if ep % ckpt_every == 0 or ep == epochs:
            ckpt_path = os.path.join(run_dir, "checkpoints", f"vae_epoch{ep:03d}.pt")
            torch.save({"model": model.state_dict(), "epoch": ep, "cfg": cfg}, ckpt_path)
            save_recon_grid(model, test_loader, os.path.join(run_dir, "plots", f"recon_ep{ep:03d}.png"))

    save_json(os.path.join(run_dir, "vae_history.json"), hist)
    return model, hist

vae, vae_hist = train_vae()
print("done")

VAE ep1:   0%|          | 0/469 [00:00<?, ?it/s]

VAE ep2:   0%|          | 0/469 [00:00<?, ?it/s]

VAE ep3:   0%|          | 0/469 [00:00<?, ?it/s]

VAE ep4:   0%|          | 0/469 [00:00<?, ?it/s]

VAE ep5:   0%|          | 0/469 [00:00<?, ?it/s]

...

VAE ep17:   0%|          | 0/469 [00:00<?, ?it/s]

VAE ep18:   0%|          | 0/469 [00:00<?, ?it/s]

VAE ep19:   0%|          | 0/469 [00:00<?, ?it/s]

VAE ep20:   0%|          | 0/469 [00:00<?, ?it/s]

done


In [9]:
class DigitCNN(nn.Module):
    # CNN classifier for MNIST digits; can also return an intermediate feature vector f(x).
    def __init__(self, feat_dim: int = 128):
        # 1x28x28 -> (Conv32+ReLU+Pool) -> 32x14x14 -> (Conv64+ReLU+Pool) -> 64x7x7 -> FC(feat_dim) -> FC(10)
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 28->14
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 14->7
        )
        self.fc1 = nn.Linear(64*7*7, feat_dim)  # feature vector f(x)
        self.fc2 = nn.Linear(feat_dim, 10)  # digit logits

    def forward(self, x, return_feat: bool = False):
        # Runs the classifier; optionally returns (logits, features) for probing.
        h = self.conv(x).flatten(1) # extract conv features
        feat = F.relu(self.fc1(h))  # f(x)
        logits = self.fc2(feat) # class scores
        return (logits, feat) if return_feat else logits # optionally return features for probing

@torch.no_grad()
def eval_digit_acc(model, loader) -> float:
    # Computes classification accuracy of the digit model on a dataloader.
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / max(1, total)

def train_classifier():
    # Trains DigitCNN on MNIST, logs loss/accuracy curves, and saves the best checkpoint.
    model = DigitCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=float(cfg["clf"]["lr"]))
    epochs = int(cfg["clf"]["epochs"])

    hist = {"train_loss": [], "val_acc": []}
    best = -1.0
    best_path = os.path.join(run_dir, "checkpoints", "clf_best.pt")

    for ep in range(1, epochs + 1):
        model.train()
        tot = 0.0
        for x, y in tqdm(train_loader, desc=f"CLF ep{ep}", leave=False):
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model(x), y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.item()

        acc = eval_digit_acc(model, test_loader)
        hist["train_loss"].append(tot / len(train_loader))
        hist["val_acc"].append(acc)

        if acc > best:
            best = acc
            torch.save({"model": model.state_dict(), "best_acc": best, "cfg": cfg}, best_path)

        plt.figure()
        plt.plot(hist["train_loss"], label="train_loss")
        plt.plot(hist["val_acc"], label="val_acc")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(run_dir, "plots", "clf_curve.png"), dpi=150)
        plt.close()

    save_json(os.path.join(run_dir, "clf_history.json"), hist)
    return model, hist, best

# Train digit classifier and print best validation accuracy.
clf, clf_hist, clf_best_acc = train_classifier()
print("clf_best_acc:", clf_best_acc)

CLF ep1:   0%|          | 0/469 [00:00<?, ?it/s]

CLF ep2:   0%|          | 0/469 [00:00<?, ?it/s]

CLF ep3:   0%|          | 0/469 [00:00<?, ?it/s]

CLF ep4:   0%|          | 0/469 [00:00<?, ?it/s]

CLF ep5:   0%|          | 0/469 [00:00<?, ?it/s]

clf_best_acc: 0.9909


In [10]:
class ThicknessProbe(nn.Module):
    # Small MLP probe: takes feature vector f(x) -> predicts thickness label (thin vs thick).
    def __init__(self, in_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), # compress features
            nn.ReLU(inplace=True),
            nn.Linear(64, 2), # 2 classes: 0 thin, 1 thick
        )

    def forward(self, feat):
         # Forward on features (not raw images).
        return self.net(feat)

@torch.no_grad()
def eval_probe_acc(clf_model, probe_model, loader, thin_k: int, thick_k: int) -> float:
    # Evaluates probe accuracy: augment images -> extract features with frozen classifier -> classify thickness.
    clf_model.eval(); probe_model.eval()
    ys, ps = [], []
    for x, _ in loader:
        x = x.to(device)
        x_aug, y = apply_thickness_aug(x, thin_k=thin_k, thick_k=thick_k)
        _, feat = clf_model(x_aug, return_feat=True)
        pred = probe_model(feat).argmax(1).detach().cpu().numpy()
        ys.extend(y.detach().cpu().numpy().tolist())
        ps.extend(pred.tolist())
    return accuracy_score(ys, ps)

def train_probe(clf_model):
    # Trains thickness probe on top of frozen digit classifier features; saves best checkpoint + curve plot.
    # freeze classifier
    for p in clf_model.parameters():
        p.requires_grad = False
    clf_model.eval()

    probe = ThicknessProbe(in_dim=128).to(device)
    opt = torch.optim.Adam(probe.parameters(), lr=float(cfg["probe"]["lr"]))
    epochs = int(cfg["probe"]["epochs"])
    thin_k = int(cfg["probe"]["aug"]["thin_k"])
    thick_k = int(cfg["probe"]["aug"]["thick_k"])
    blend = int(cfg["probe"]["aug"]["blend"])

    hist = {"train_loss": [], "val_acc": []}
    best = -1.0
    best_path = os.path.join(run_dir, "checkpoints", "probe_best.pt")

    for ep in range(1, epochs + 1):
        probe.train()
        tot = 0.0
        for x, _ in tqdm(train_loader, desc=f"PROBE ep{ep}", leave=False):
            x = x.to(device)
            x_aug, y = apply_thickness_aug(x, thin_k=thin_k, thick_k=thick_k, blend=blend)
            _, feat = clf_model(x_aug, return_feat=True)
            loss = F.cross_entropy(probe(feat), y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.item()

        acc = eval_probe_acc(clf_model, probe, test_loader, thin_k, thick_k)
        hist["train_loss"].append(tot / len(train_loader))
        hist["val_acc"].append(acc)

        if acc > best:
            best = acc
            torch.save({"model": probe.state_dict(), "best_acc": best, "cfg": cfg}, best_path)

        plt.figure()
        plt.plot(hist["train_loss"], label="train_loss")
        plt.plot(hist["val_acc"], label="val_acc")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(run_dir, "plots", "probe_curve.png"), dpi=150)
        plt.close()

    save_json(os.path.join(run_dir, "probe_history.json"), hist)
    return probe, hist, best

probe, probe_hist, probe_best_acc = train_probe(clf)
print("probe_best_acc:", probe_best_acc)

PROBE ep1:   0%|          | 0/469 [00:00<?, ?it/s]

PROBE ep2:   0%|          | 0/469 [00:00<?, ?it/s]

PROBE ep3:   0%|          | 0/469 [00:00<?, ?it/s]

PROBE ep4:   0%|          | 0/469 [00:00<?, ?it/s]

PROBE ep5:   0%|          | 0/469 [00:00<?, ?it/s]

probe_best_acc: 0.9893


In [12]:
def write_memo(run_dir: str, cfg: dict, row: dict):
    memo = f"""# Week 5 — Weekly Update (1 page)

## Goal
Make the baseline fully working & reproducible: Conv-VAE (MNIST, latent dim = 16) + digit classifier (content preservation + feature extractor f(x)) + stroke-thickness probe s(x) with deterministic runs and saved artifacts.

## What I ran
- seed: {cfg['seed']}
- device: {cfg['device']} (actual: {device})
- VAE: z={cfg['vae']['latent_dim']}, beta={cfg['vae']['beta']}, epochs={cfg['vae']['epochs']}
- CLF epochs={cfg['clf']['epochs']}
- PROBE epochs={cfg['probe']['epochs']} thin_k={cfg['probe']['aug']['thin_k']} thick_k={cfg['probe']['aug']['thick_k']}

## Results (key)
- clf_best_acc: {row['clf_best_acc']:.4f}
- probe_best_acc: {row['probe_best_acc']:.4f}
- vae_last_val_loss: {row['vae_last_val_loss']:.4f}

## Key plots
- plots/vae_loss_curve.png
- plots/recon_ep*.png
- plots/clf_curve.png
- plots/probe_curve.png
- plots/thickness_aug_preview.png

## Quick interpretation
- VAE: loss decreases smoothly; train/val curves stay close; reconstructions are stable and recognizable by ~epoch 5–10.
- Classifier: near-99% MNIST accuracy; usable as a content-preservation signal and as f(x).
- Probe: high accuracy on thin vs thick; next step is to verify it is not solving the task via a trivial “ink area” shortcut only.

## Next week plan (Week 6)
- Learn global steering directions in latent space and evaluate: probe score ↑ vs digit accuracy ↓ (content preservation trade-off).
- Add a cleaner evaluation protocol:
  - digit_acc on original vs augmented vs steered reconstructions
  - probe score on reconstructions (not only on augmented inputs)
- Add one sanity metric for thickness augmentation (e.g., mean ink_gap between thin/thick).

## Risks / issues
- Probe shortcut risk: it may rely on a simple global statistic (e.g., total “ink area”) rather than thickness structure → mitigate by randomizing augmentation strength per sample and monitoring ink_gap + digit accuracy.
- Over-steering risk (next week): improving thickness score may degrade digit identity → track digit_acc/f(x) similarity.
- Notebook environment: DataLoader multiprocessing can be noisy in some setups → prefer num_workers=0 or persistent spawn workers for stability.
"""
    with open(os.path.join(run_dir, "memo_1page.md"), "w", encoding="utf-8") as f:
        f.write(memo)

row = {
    "run_dir": run_dir,
    "seed": int(cfg["seed"]),
    "device_cfg": cfg["device"],
    "device_actual": str(device),
    "vae_latent_dim": int(cfg["vae"]["latent_dim"]),
    "vae_beta": float(cfg["vae"]["beta"]),
    "vae_last_val_loss": float(vae_hist["val_loss"][-1]),
    "clf_best_acc": float(clf_best_acc),
    "probe_best_acc": float(probe_best_acc),
}
pd.DataFrame([row]).to_csv(os.path.join(run_dir, "results.csv"), index=False)
write_memo(run_dir, cfg, row)

print("Saved:", os.path.join(run_dir, "results.csv"))
print("Saved:", os.path.join(run_dir, "memo_1page.md"))

Saved: runs/20260224_074746_week5/results.csv
Saved: runs/20260224_074746_week5/memo_1page.md


In [14]:
!zip -r output.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/runs/ (stored 0%)
  adding: kaggle/working/runs/20260224_074746_week5/ (stored 0%)
  adding: kaggle/working/runs/20260224_074746_week5/clf_history.json (deflated 43%)
  adding: kaggle/working/runs/20260224_074746_week5/probe_history.json (deflated 43%)
  adding: kaggle/working/runs/20260224_074746_week5/checkpoints/ (stored 0%)
  adding: kaggle/working/runs/20260224_074746_week5/checkpoints/vae_epoch015.pt (deflated 7%)
  adding: kaggle/working/runs/20260224_074746_week5/checkpoints/vae_epoch005.pt (deflated 7%)
  adding: kaggle/working/runs/20260224_074746_week5/checkpoints/vae_epoch020.pt (deflated 7%)
  adding: kaggle/working/runs/20260224_074746_week5/checkpoints/probe_best.pt (deflated 11%)
  adding: kaggle/working/runs/20260224_074746_week5/checkpoints/vae_epoch010.pt (deflated 7%)
  adding: kaggle/working/runs/20260224_074746_week5/checkpoints/clf_best.pt (deflated 7%)
  adding: kaggle/working/runs/20260224_074746_we